In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025E1EB47A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025E1ED74590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
from pydantic import BaseModel,Field

class movie(BaseModel):
    title:str=Field(description="The Title of the movie")
    year:int=Field(description="The year when movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")

In [11]:
model_with_structured_output = model.with_structured_output(movie)
model_with_structured_output

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025E1EB47A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025E1ED74590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The Title of the movie', 'type': 'string'}, 'year': {'description': 'The year when movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie

In [12]:
model.invoke("Provide details of the movie inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 movie directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. The genre is sci-fi with elements of action and thriller. The story revolves around a concept called "inception," which is the act of planting an idea into someone\'s subconscious. \n\nThe protagonist is Dom Cobb, played by DiCaprio, a professional thief who steals secrets by infiltrating the subconscious of his targets. He\'s offered a chance to erase his criminal past by performing the inverse: planting an idea instead of stealing it. The challenge is to perform this heist within the dream world, which requires navigating different layers of dreams. \n\nThere\'s a lot about the dream-sharing technology. The team uses a device called the PASIV device, which allows them to enter a subject\'s dream. They have 

In [14]:
response = model_with_structured_output.invoke("Provide details of the movie inception")
response

movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [15]:
from pydantic import BaseModel,Field

class movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="The Title of the movie")
    year:int=Field(...,description="The year when movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie out of 10")

model_with_structured_output1 = model.with_structured_output(movie,include_raw=True)

response = model_with_structured_output1.invoke("Provide details of movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a function called \'movie\' that requires title, year, director, and rating. I need to make sure I have all that information for Inception. The title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does have it at 8.8. So I need to structure the tool call with those parameters. Make sure all required fields are included. No need for any optional parameters here. Alright, that should cover it.\n', 'tool_calls': [{'id': '2g93g6yv9', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 199, 'prompt_tokens': 230, 'total_tokens': 

### NEsted Structure

In [16]:
from pydantic import BaseModel,Field

class actor(BaseModel):
    name:str
    role:str

class movie(BaseModel):
    title:str
    year:int
    cast:list[actor]
    genres:list[str]
    budget:float | None = Field(description="Budget in millions USD")

movie_with_structured_output3 = model.with_structured_output(movie)
response = movie_with_structured_output3.invoke("Provide details about the movie Inception")
response


movie(title='Inception', year=2010, cast=[actor(name='Leonardo DiCaprio', role='Dom Cobb'), actor(name='Joseph Gordon-Levitt', role='Arthur'), actor(name='Ellen Page', role='Ariadne'), actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

## Type dict

In [17]:
from typing_extensions import TypedDict,Annotated

class MovieDetails(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"The year movie was released"]
    director: Annotated[str,...,"The director of the movie"]
    rating: Annotated[float,...,"The rating of the movie out of 10"]

movie_withtypeddict = model.with_structured_output(MovieDetails)
response = movie_withtypeddict.invoke("Give me the movie details of Shining")
response

{'director': 'Stanley Kubrick',
 'rating': 8.1,
 'title': 'The Shining',
 'year': 1980}

In [22]:
from typing_extensions import TypedDict,Annotated

class actor(TypedDict):
    name:str
    role:str

class movie(TypedDict):
    title:str
    year:int
    cast:list[actor]
    genres:list[str]
    budget:float | None = Field(description="Budget in millions USD")

movie_with_structured_output3 = model.with_structured_output(movie)
response = movie_with_structured_output3.invoke("Provide details about the movie Inception")
response


{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'James'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [24]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## Dataclass

In [30]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class contactinfo(BaseModel):
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email of the person")
    phone:int=Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=contactinfo
)    

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='265ef7d7-9476-4c12-a469-b2f4a501017d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". \n\nFirst, I need to identify the different parts. The name is straightforward, "John Doe". Then there\'s the email address, which is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function parameters require the phone number as an integer, but the input has parentheses and a space. Wait, the phone number here is (555) 123-4567. The function expects an integer, so I need to remove the non-numeric characters. Let me process that.\n\nSo, the phone number would be 5551234567. I\'ll make sure to convert it to an integer without

In [31]:
result["structured_response"]

contactinfo(name='John Doe', email='john@example.com', phone=5551234567)

In [32]:
from langchain.agents import create_agent

class contactinfo(TypedDict):
    name:str
    email:str
    phone:int

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=contactinfo
)    

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': 5551234567}

In [33]:
from dataclasses import dataclass

@dataclass
class contactinfo:
    """Contact info of a person"""
    name:str
    email:str
    phone:int 


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=contactinfo
)    

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

})

result['structured_response']


contactinfo(name='John Doe', email='john@example.com', phone=5551234567)